In [ ]:
"""
TextLoader会将一个文档文件加载为一个Document对象，该对象有两个属性：
    metadata:       存储该文档的来源路径等元数据
    page_content:   存储文档的内容
"""

from langchain_community.document_loaders import TextLoader

text_documents = TextLoader("knowledge_base/sample.txt", encoding="utf-8").load()
print(text_documents)


In [ ]:
"""
UnstructuredMarkdownLoader可用于加载Markdown文件
    mode: 加载模式
        "single"    返回单个Document对象
        "elements"  按标题等元素切分文档
    strategy: 加载策略
        "fast"      快速粗粒度加载
        "hi_res"    细粒度加载，按标题层级、列表项、表格等结构细分
"""

from langchain_community.document_loaders import UnstructuredMarkdownLoader

md_documents = UnstructuredMarkdownLoader(
    "knowledge_base/sample.md",
    # mode="elements",
    mode="single",
    strategy="fast",
).load()
print(md_documents)


In [ ]:
"""
PyPDFLoader
    支持页级拆分，每一页作为一个Document返回
    支持提取图像、提取布局
    extraction_mode: 提取模式
        "plain"     提取纯文本
        "layout"    提取布局
"""

from langchain_community.document_loaders import PyPDFLoader

pdf_documents = PyPDFLoader(
    "knowledge_base/sample.pdf",
    # extraction_mode="layout",
    extraction_mode="plain",
).load()
print(pdf_documents)


In [ ]:
"""
UnstructuredPDFLoader
    支持结构化提取，支持OCR
    仅当 PDF 文档中不存在文本时，才会应用 OCR
    mode: 加载模式
        "single"    返回单个Document对象
        "elements"  按标题等元素切分文档
    strategy: 加载策略
        "fast"      提取并处理文本
        "ocr_only"  先进行 OCR 处理，再处理原始文本
        "hi_res"    识别文档布局并处理，自动下载YOLOX模型（识别页面布局）
    infer_table_structure: 是否推断表格结构
        仅 hi_res 下起效
        如果为 True，提取出的表格元素会包含一个 text_as_html 元数据，将文本内容转换为 html 格式
    languages: OCR使用的语言
        需传入语言列表
        语言列表参考 https://github.com/tesseract-ocr/langdata
    更多参数详见 https://github.com/Unstructured-IO/unstructured/blob/main/unstructured/partition/pdf.py
"""

from langchain_community.document_loaders import UnstructuredPDFLoader

pdf_documents = UnstructuredPDFLoader(
    "knowledge_base/sample.pdf",
    mode="elements",
    strategy="hi_res",
    infer_table_structure=True,
    languages=["eng", "chi_sim"],
).load()
print(pdf_documents)


In [ ]:
"""
UnstructuredWordDocumentLoader
    适用于 .docx 和 .doc 文件
    mode: 加载模式
        "single"    返回单个Document对象
        "elements"  按标题等元素切分文档
    strategy: 加载策略
        "fast"      快速粗粒度加载
        "hi_res"    细粒度加载，按结构细分
"""

from langchain_community.document_loaders import UnstructuredWordDocumentLoader

word_documents = UnstructuredWordDocumentLoader(
    "knowledge_base/sample.docx",
    mode="elements",
    strategy="fast",
).load()
print(word_documents)


In [ ]:
import re
import json

def clean_content(documents: list):

    """文本清洗"""
    cleaned_docs = []


    for doc in documents:

        # 1、page_content处理：去除多余换行和空格
        text = doc.page_content

        # 将连续的换行符替换为两个换行符，正则表达式模式：r"\n{2,}"
        # r"" 表示原始字符串（raw string），避免转义字符的特殊处理
        # \n 表示换行符
        # {2,} 是量词，表示前面的字符（换行符）出现 2 次或更多次
        text = re.sub(r"\n{2,}", "\n\n", text)
        text = text.strip()

        # 2、metadata处理：将所有非 Chroma 支持类型的值转为 JSON 格式字符串
        allowed_types = (str, int, float, bool)
        for key, value in doc.metadata.items():
            if not isinstance(value, allowed_types):
                try:
                    doc.metadata[key] = json.dumps(value, ensure_ascii=False)
                except (TypeError, ValueError):
                    # 如果 json.dumps 失败（如包含不可序列化对象），转为 str
                    doc.metadata[key] = str(value)

        # 3、更新文档内容
        doc.page_content = text
        cleaned_docs.append(doc)

    return cleaned_docs


In [ ]:
from langchain_community.document_loaders import (
    TextLoader,
    UnstructuredMarkdownLoader,
    UnstructuredPDFLoader,
    UnstructuredWordDocumentLoader
)

def load_documents():
    """
    加载多种类型的文档，包括text、markdown、PDF和Word文档

    Returns:
        list: 包含所有加载文档的列表
    """
    # 加载文本文件
    text_documents = TextLoader(
        "knowledge_base/sample.txt",
        encoding="utf8"
    ).load()

    # 加载Markdown文件
    md_documents = UnstructuredMarkdownLoader(
        "knowledge_base/sample.md"
    ).load()

    # 加载PDF文件
    pdf_documents = UnstructuredPDFLoader(
        "knowledge_base/sample.pdf",
        mode="elements",  # 元素模式
        strategy="hi_res",  # 高分辨率策略
        # strategy="fast",
        languages=["eng", "chi_sim"],  # 支持的语言：英文和简体中文
    ).load()

    # 加载Word文档
    word_documents = UnstructuredWordDocumentLoader(
        "knowledge_base/sample.docx"
    ).load()

    # 返回所有文档的列表
    return text_documents + md_documents + pdf_documents + word_documents




In [ ]:
documents = load_documents()
documents = clean_content(documents)
print(documents)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 文本分块
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "。"],  # 分隔符列表
    chunk_size=500,  # 每个块的最大长度
    chunk_overlap=50,  # 每个块重叠的长度
)
texts = text_splitter.split_documents(documents)


In [ ]:
print(texts)

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

# 加载嵌入模型
embedding_model = HuggingFaceEmbeddings(
    model_name="/root/embedding_model/BAAI/bge-base-zh-v1.5",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={
        "normalize_embeddings": True
    },  # 输出归一化向量，更适合余弦相似度计算
)

# 从 Document 中取出文本
page_content_list = [text.page_content for text in texts]
# 进行嵌入
embeddings = embedding_model.embed_documents(page_content_list)
# 打印嵌入结果
for i, (page_content, vector) in enumerate(zip(page_content_list, embeddings)):
    print("Text:\n", page_content)
    print("Embedding:\n", vector[:5])
    print()
    if i == 5:
        break


In [ ]:
from langchain_chroma import Chroma

# 嵌入并存储到向量数据库
vectorstore = Chroma.from_documents(
    texts,  # 文档列表
    embedding_model,  # 嵌入模型
    persist_directory="vectorstore",  # 存储路径
)


In [ ]:
print(vectorstore.get().keys())  # 查看所有属性
print(vectorstore.get(include=["embeddings"])["embeddings"][:5, :5])  # 查看嵌入向量


# 向已存在的Chroma中新增数据

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

# 加载嵌入模型
embedding_model = HuggingFaceEmbeddings(
    model_name="/root/embedding_model/BAAI/bge-base-zh-v1.5",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={
        "normalize_embeddings": True
    },  # 输出归一化向量，更适合余弦相似度计算
)

from langchain_chroma import Chroma
# 初始化 Chroma 客户端
vectorstore = Chroma(
    persist_directory="vectorstore",
    embedding_function=embedding_model,
)


In [ ]:
"""
WebBaseLoader
    适用于网页
    web_paths: 网址序列
    bs_kwargs: 传给 BeautifulSoup 的解析参数
        parse_only  只提取指定标签的元素
"""
import bs4
from langchain_community.document_loaders import WebBaseLoader

web_documents = WebBaseLoader(
    web_paths=(
        "https://news.sina.com.cn/c/xl/2025-09-07/doc-infprmwn0510979.shtml",
    ),
    bs_kwargs={"parse_only": bs4.SoupStrainer(id="article")},  # 只提取正文区域
).load()
print(web_documents)


In [ ]:
# 向chroma插入新的document
vectorstore.add_documents(web_documents)


In [ ]:
vectorstore.get()

In [ ]:
specific_documents=vectorstore.get(where_document={"$contains": "纪念中国人民"})
print(specific_documents)

In [ ]:
vectorstore.delete(ids='2accd90d-77fe-4a21-9159-26a876a1b69f')

In [ ]:
vectorstore.get()
# specific_documents=vectorstore.get(where_document={"$contains": "纪念中国人民"})
# print(specific_documents)